# 🧠 Multimodal Depression Detection — Cross-Attention Fusion
Combines 128-d Facial (BiLSTM) + 768-d Text (RoBERTa) embeddings
using a Cross-Attention layer for final PHQ-8 prediction.

**Run cells in order: Cell 0 → 1 → 2 → 3 → 4**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 0 — Setup & Config
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F_func
import numpy as np
import pandas as pd
import os, json
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, mean_absolute_error, mean_squared_error,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# ── Paths ──────────────────────────────────────────────────────
DRIVE_DIR   = '/content/drive/MyDrive/edaic'
FUSION_DIR  = f'{DRIVE_DIR}/fusion_inputs'
LABELS_CSV  = f'{DRIVE_DIR}/facial_data/labels.csv'
WORK_DIR    = '/content/fusion_work'
os.makedirs(WORK_DIR, exist_ok=True)

# ── Hyperparameters ─────────────────────────────────────────────
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED        = 42
N_FOLDS     = 5
BATCH_SIZE  = 32
EPOCHS      = 60
PATIENCE    = 12
LR          = 1e-4
D_MODEL     = 256   # Fusion projection dimension
N_HEADS     = 4     # Cross-attention heads

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device : {DEVICE}')
print(f'Config : {EPOCHS} epochs, {N_FOLDS}-fold CV, batch={BATCH_SIZE}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Load & Align Embeddings
# ═══════════════════════════════════════════════════════════════

# Load embeddings
face_emb  = np.load(f'{FUSION_DIR}/facial_embeddings_all.npy')    # (N, 128)
text_emb  = np.load(f'{FUSION_DIR}/text_embeddings_all.npy')      # (N, 768)
pids_face = np.load(f'{FUSION_DIR}/participant_ids_facial.npy').astype(int)
pids_text = np.load(f'{FUSION_DIR}/participant_ids_text.npy').astype(int)

print(f'Loaded:  Face {face_emb.shape}  |  Text {text_emb.shape}')
print(f'IDs:     Face {len(pids_face)}  |  Text {len(pids_text)}')

# Align by participant ID (inner join)
common_pids = sorted(set(pids_face) & set(pids_text))
print(f'Common participants: {len(common_pids)}')

face_idx = {p: i for i, p in enumerate(pids_face)}
text_idx = {p: i for i, p in enumerate(pids_text)}

aligned_face = np.array([face_emb[face_idx[p]] for p in common_pids])
aligned_text = np.array([text_emb[text_idx[p]] for p in common_pids])
aligned_pids = np.array(common_pids)

# Load labels
labels_df = pd.read_csv(LABELS_CSV)
labels_df = labels_df.set_index('Participant_ID')

y_phq = np.array([labels_df.loc[p, 'PHQ_Score'] for p in aligned_pids], dtype=np.float32)
y_bin = np.array([labels_df.loc[p, 'PHQ_Binary'] for p in aligned_pids], dtype=np.int64)

def phq_to_sev(s):
    if s <= 4:  return 0
    if s <= 9:  return 1
    if s <= 14: return 2
    if s <= 19: return 3
    return 4

y_sev = np.array([phq_to_sev(s) for s in y_phq], dtype=np.int64)
NUM_CLASSES = len(np.unique(y_sev))

# Validate variance
print(f'\nEmbedding variance check:')
print(f'  Face std : {aligned_face.std():.4f}  (should be > 0.01)')
print(f'  Text std : {aligned_text.std():.4f}  (should be > 0.01)')
print(f'\nLabels    : PHQ range [{y_phq.min():.0f}, {y_phq.max():.0f}]  |  Classes: {NUM_CLASSES}')
sev_names = ['None','Mild','Moderate','Mod-Sev','Severe']
unique, counts = np.unique(y_sev, return_counts=True)
print(f'Severity distribution:')
for u, c in zip(unique, counts):
    print(f'  {sev_names[u]:12s}: {c}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Model Architecture
# ═══════════════════════════════════════════════════════════════

class FusionDataset(Dataset):
    def __init__(self, face, text, yp, ys):
        self.face = torch.FloatTensor(face)
        self.text = torch.FloatTensor(text)
        self.yp   = torch.FloatTensor(yp)
        self.ys   = torch.LongTensor(ys)
    def __len__(self): return len(self.face)
    def __getitem__(self, i):
        return self.face[i], self.text[i], self.yp[i], self.ys[i]


class CrossAttentionFusion(nn.Module):
    """
    Cross-Attention Fusion:
      - Text (768-d) as Query  → what does the language 'look for'?
      - Face (128-d) as Key/Value → visual evidence to attend to
    """
    def __init__(self, d_face=128, d_text=768, d_model=256,
                 n_heads=4, n_classes=5, dropout=0.3):
        super().__init__()

        # Projection: bring both modalities to d_model
        self.face_proj = nn.Sequential(
            nn.Linear(d_face, d_model),
            nn.LayerNorm(d_model),
            nn.GELU()
        )
        self.text_proj = nn.Sequential(
            nn.Linear(d_text, d_model),
            nn.LayerNorm(d_model),
            nn.GELU()
        )

        # Cross-Attention: Text queries Face
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        # Self-Attention on Text (enrich text representation)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        # Post-fusion normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

        # Fusion MLP: combines both streams
        fused_dim = d_model * 2    # cross_attn_out + face_proj
        self.fusion_mlp = nn.Sequential(
            nn.Linear(fused_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU()
        )

        final_dim = d_model // 2

        # Output heads
        self.regr_head = nn.Linear(final_dim, 1)          # PHQ-8 regression
        self.cls_head  = nn.Linear(final_dim, n_classes)  # Severity classification

    def forward(self, face, text):
        # Project to common dimension
        f = self.face_proj(face).unsqueeze(1)   # (B, 1, D)
        t = self.text_proj(text).unsqueeze(1)   # (B, 1, D)

        # Cross-Attention: Text (Q) attends to Face (K, V)
        ca_out, _ = self.cross_attn(t, f, f)
        ca_out = self.norm1(t + ca_out)          # Residual + Norm (B, 1, D)

        # Concat cross-attn output with face projection
        combined = torch.cat([
            ca_out.squeeze(1),
            f.squeeze(1)
        ], dim=-1)  # (B, D*2)

        # Final fusion MLP
        fused = self.fusion_mlp(combined)       # (B, D//2)
        fused = self.drop(fused)

        phq_pred = self.regr_head(fused).squeeze(-1)  # (B,)
        sev_pred = self.cls_head(fused)               # (B, n_classes)
        return phq_pred, sev_pred, fused


class FusionLoss(nn.Module):
    def __init__(self, cls_weight=0.5):
        super().__init__()
        self.mse = nn.MSELoss()
        self.ce  = nn.CrossEntropyLoss()
        self.alpha = cls_weight
    def forward(self, phq_pred, phq_true, sev_pred, sev_true):
        return self.mse(phq_pred, phq_true) + self.alpha * self.ce(sev_pred, sev_true)


def make_sampler(ys):
    cls, cnt = np.unique(ys, return_counts=True)
    w = torch.FloatTensor([1.0 / cnt[list(cls).index(s)] for s in ys])
    return WeightedRandomSampler(w, len(w))


# Quick sanity check
dummy_face = torch.randn(4, 128)
dummy_text = torch.randn(4, 768)
test_model = CrossAttentionFusion(n_classes=NUM_CLASSES)
p, s, e = test_model(dummy_face, dummy_text)
print(f'✅ Model OK  →  PHQ: {p.shape}  Severity: {s.shape}  Embedding: {e.shape}')
n_params = sum(par.numel() for par in test_model.parameters() if par.requires_grad)
print(f'   Trainable parameters: {n_params:,}')
del test_model

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — 5-Fold Cross-Validation Training
# ═══════════════════════════════════════════════════════════════

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_results = []
best_global_mae = 999
best_fold_num   = -1

for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(aligned_pids, y_sev)):
    print(f'\n{"="*50}')
    print(f'FOLD {fold_idx+1} / {N_FOLDS}')
    print(f'{"="*50}')

    # ── Data ────────────────────────────────────────────────────
    face_tr, text_tr = aligned_face[tr_idx], aligned_text[tr_idx]
    face_va, text_va = aligned_face[va_idx], aligned_text[va_idx]
    yp_tr, ys_tr = y_phq[tr_idx], y_sev[tr_idx]
    yp_va, ys_va = y_phq[va_idx], y_sev[va_idx]

    train_ds = FusionDataset(face_tr, text_tr, yp_tr, ys_tr)
    val_ds   = FusionDataset(face_va, text_va, yp_va, ys_va)

    train_ldr = DataLoader(train_ds, BATCH_SIZE,
                           sampler=make_sampler(ys_tr))
    val_ldr   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False)

    # ── Model, Optimizer, Scheduler ─────────────────────────────
    model = CrossAttentionFusion(
        d_face=128, d_text=768,
        d_model=D_MODEL, n_heads=N_HEADS,
        n_classes=NUM_CLASSES, dropout=0.3
    ).to(DEVICE)

    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
    crit   = FusionLoss(cls_weight=0.5)

    best_mae  = 999
    no_improve = 0
    history   = {'train_loss': [], 'val_mae': []}

    # ── Training Loop ───────────────────────────────────────────
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        for face_b, text_b, yp_b, ys_b in train_ldr:
            face_b = face_b.to(DEVICE)
            text_b = text_b.to(DEVICE)
            yp_b   = yp_b.to(DEVICE)
            ys_b   = ys_b.to(DEVICE)

            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
                pp, sp, _ = model(face_b, text_b)
                loss = crit(pp, yp_b, sp, ys_b)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            epoch_loss += loss.item()

        sched.step()

        # ── Validation ──────────────────────────────────────────
        model.eval()
        val_pp, val_pt = [], []
        with torch.no_grad():
            for face_b, text_b, yp_b, ys_b in val_ldr:
                pp, _, _ = model(face_b.to(DEVICE), text_b.to(DEVICE))
                val_pp += np.clip(pp.cpu().numpy(), 0, 27).tolist()
                val_pt += yp_b.tolist()

        val_mae = mean_absolute_error(val_pt, val_pp)
        history['train_loss'].append(epoch_loss / len(train_ldr))
        history['val_mae'].append(val_mae)

        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d} | Loss {epoch_loss/len(train_ldr):.3f} | Val MAE {val_mae:.3f}')

        # ── Early Stopping ──────────────────────────────────────
        if val_mae < best_mae:
            best_mae = val_mae
            no_improve = 0
            torch.save(model.state_dict(), f'{WORK_DIR}/fusion_fold{fold_idx}.pt')
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    # ── Full Val Metrics ────────────────────────────────────────
    model.load_state_dict(torch.load(f'{WORK_DIR}/fusion_fold{fold_idx}.pt'))
    model.eval()
    pp_all, pt_all, sp_all, st_all = [], [], [], []
    with torch.no_grad():
        for face_b, text_b, yp_b, ys_b in val_ldr:
            pp, sp, _ = model(face_b.to(DEVICE), text_b.to(DEVICE))
            pp_all += np.clip(pp.cpu().numpy(), 0, 27).tolist()
            pt_all += yp_b.tolist()
            sp_all += sp.argmax(-1).cpu().tolist()
            st_all += ys_b.tolist()

    mae  = mean_absolute_error(pt_all, pp_all)
    rmse = np.sqrt(mean_squared_error(pt_all, pp_all))
    f1_5 = f1_score(st_all, sp_all, average='macro', zero_division=0)
    f1_b = f1_score(
        (np.array(pt_all) >= 10).astype(int),
        (np.array(pp_all) >= 10).astype(int),
        average='binary', zero_division=0
    )

    print(f'\nFOLD {fold_idx+1} RESULTS → MAE={mae:.3f}  RMSE={rmse:.3f}  F1-5={f1_5:.3f}  F1-bin={f1_b:.3f}')
    fold_results.append({'fold': fold_idx+1, 'mae': mae, 'rmse': rmse, 'f1_5': f1_5, 'f1_bin': f1_b})

    if mae < best_global_mae:
        best_global_mae = mae
        best_fold_num   = fold_idx
        import shutil
        shutil.copy(f'{WORK_DIR}/fusion_fold{fold_idx}.pt', f'{WORK_DIR}/fusion_best.pt')

# ── Summary ─────────────────────────────────────────────────────
print(f'\n{"="*50}')
print('CROSS-VALIDATION SUMMARY')
print(f'{"="*50}')
df_res = pd.DataFrame(fold_results)
print(df_res.to_string(index=False))
print(f'\nMAE   : {df_res.mae.mean():.3f} ± {df_res.mae.std():.3f}')
print(f'RMSE  : {df_res.rmse.mean():.3f} ± {df_res.rmse.std():.3f}')
print(f'F1-5  : {df_res.f1_5.mean():.3f} ± {df_res.f1_5.std():.3f}')
print(f'F1-bin: {df_res.f1_bin.mean():.3f} ± {df_res.f1_bin.std():.3f}')
print(f'\nBest fold: {best_fold_num + 1}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Final Evaluation + Plots + Save to Drive
# ═══════════════════════════════════════════════════════════════

# Load best model and run on ALL data for final metrics/plots
best_model = CrossAttentionFusion(
    d_face=128, d_text=768,
    d_model=D_MODEL, n_heads=N_HEADS,
    n_classes=NUM_CLASSES, dropout=0.3
).to(DEVICE)
best_model.load_state_dict(torch.load(f'{WORK_DIR}/fusion_best.pt'))
best_model.eval()
print('✅ Best model loaded')

full_ds  = FusionDataset(aligned_face, aligned_text, y_phq, y_sev)
full_ldr = DataLoader(full_ds, BATCH_SIZE, shuffle=False)

pp_all, pt_all, sp_all, st_all, emb_all = [], [], [], [], []
with torch.no_grad():
    for face_b, text_b, yp_b, ys_b in full_ldr:
        pp, sp, emb = best_model(face_b.to(DEVICE), text_b.to(DEVICE))
        pp_all  += np.clip(pp.cpu().numpy(), 0, 27).tolist()
        pt_all  += yp_b.tolist()
        sp_all  += sp.argmax(-1).cpu().tolist()
        st_all  += ys_b.tolist()
        emb_all.append(emb.cpu().numpy())

pp_all   = np.array(pp_all)
pt_all   = np.array(pt_all)
sp_all   = np.array(sp_all)
st_all   = np.array(st_all)
fused_emb = np.vstack(emb_all)

# ── Metrics ─────────────────────────────────────────────────────
mae   = mean_absolute_error(pt_all, pp_all)
rmse  = np.sqrt(mean_squared_error(pt_all, pp_all))
f1_5  = f1_score(st_all, sp_all, average='macro', zero_division=0)
f1_b  = f1_score((pt_all>=10).astype(int), (pp_all>=10).astype(int),
                 average='binary', zero_division=0)
sev_names = ['None','Mild','Mod.','Mod-Sev','Severe'][:NUM_CLASSES]

print(f'\n  PHQ-8 MAE          : {mae:.3f}')
print(f'  PHQ-8 RMSE         : {rmse:.3f}')
print(f'  F1-macro (5-class) : {f1_5:.3f}')
print(f'  F1 (binary depress): {f1_b:.3f}  ← KEY for paper')
print(f'\n{classification_report(st_all, sp_all, target_names=sev_names, zero_division=0)}')

# ── Plots ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Multimodal Fusion — Final Evaluation', fontsize=14, fontweight='bold')

# 1. Regression scatter
axes[0].scatter(pt_all, pp_all, alpha=0.6, s=35, c='#4c72b0', edgecolors='w', linewidth=0.4)
axes[0].plot([0, 27], [0, 27], 'r--', lw=1.5, label='Perfect')
axes[0].set_title(f'PHQ-8 Regression\nMAE={mae:.2f}  RMSE={rmse:.2f}')
axes[0].set_xlabel('True PHQ-8');  axes[0].set_ylabel('Predicted PHQ-8')
axes[0].legend()

# 2. Confusion matrix
cm = confusion_matrix(st_all, sp_all)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1],
            xticklabels=sev_names, yticklabels=sev_names)
axes[1].set_title(f'Severity Confusion Matrix\nF1-macro={f1_5:.3f}')
axes[1].set_xlabel('Predicted');  axes[1].set_ylabel('True')

# 3. t-SNE of fused embeddings
try:
    from sklearn.manifold import TSNE
    e2d = TSNE(2, perplexity=min(30, len(aligned_pids)-1), random_state=SEED).fit_transform(fused_emb)
    sc  = axes[2].scatter(e2d[:,0], e2d[:,1], c=st_all, cmap='RdYlGn_r',
                          s=45, alpha=0.85, edgecolors='w', linewidth=0.4)
    plt.colorbar(sc, ax=axes[2], label='Severity')
    axes[2].set_title('t-SNE: Fused Embeddings\n(Coloured by Depression Severity)')
except Exception as tsne_e:
    print(f't-SNE skipped: {tsne_e}')

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fusion_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Save everything to Drive ─────────────────────────────────────
SAVE_DIR = f'{DRIVE_DIR}/fusion_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)

import shutil
shutil.copy(f'{WORK_DIR}/fusion_best.pt',          f'{SAVE_DIR}/fusion_best.pt')
shutil.copy(f'{WORK_DIR}/fusion_evaluation.png',   f'{SAVE_DIR}/fusion_evaluation.png')

np.save(f'{SAVE_DIR}/fused_embeddings_all.npy',  fused_emb)
np.save(f'{SAVE_DIR}/participant_ids_final.npy', aligned_pids)

pd.DataFrame({
    'participant_id': aligned_pids,
    'phq_true':  pt_all,
    'phq_pred':  pp_all.round(2),
    'sev_true':  st_all,
    'sev_pred':  sp_all,
    'dep_true':  (pt_all >= 10).astype(int),
    'dep_pred':  (pp_all >= 10).astype(int)
}).to_csv(f'{SAVE_DIR}/fusion_predictions.csv', index=False)

df_res.to_csv(f'{SAVE_DIR}/cv_results.csv', index=False)

print(f'\n✅ All outputs saved to: {SAVE_DIR}')
print(f'\n🎉 MULTIMODAL FUSION COMPLETE!')
print(f'   MAE    = {mae:.3f}')
print(f'   RMSE   = {rmse:.3f}')
print(f'   F1-bin = {f1_b:.3f}')
print(f'   F1-5   = {f1_5:.3f}')